# PTF (Piyasa Takas Fiyatı) — Yük Eğrilerinden Arz-Talep Dengesine

**Amaç:** "Önce yük eğrilerine bak, sonra fiyat eğrilerine bak. Fiyatların arz-talep
dengesini yansıtması gerekir." — bu yorumu adım adım, veriyle test edilebilir bir
analize çeviriyoruz.

**Analizin mantığı (neden bu sırayla?):**

| Adım | Ne yapıyoruz | Neden |
|---|---|---|
| 1 | **Yük eğrileri** — talep tarafının şekli | Talep, elektrikte kısa vadede neredeyse tamamen fiyat-inelastik. Yani fiyatı belirleyen denklemde talep *dışsal bir girdi*. Önce onun ritmini (günlük/haftalık/mevsimsel) öğreniyoruz. |
| 2 | **Fiyat eğrileri** — PTF'nin şekli | Fiyatın kendi ritmi, dağılımı, kuyrukları. Yükle aynı ritmi mi paylaşıyor? |
| 3 | **Arz tarafı** — üretim kırılımı, artık yük | Fiyatı belirleyen *brüt talep değil*, sıfır marjinal maliyetli üretimden arta kalan **artık yük**. Merit order burada devreye giriyor. |
| 4 | **Denge** — ampirik arz eğrisi | PTF ile artık yükü aynı grafikte birleştirdiğimizde piyasanın **gerçek arz eğrisini** (merit order curve) çiziyoruz. Asıl "arz-talep dengesi" bu. |
| 5 | **Tahminleme köprüsü** | Buradan çıkan değişkenler, PTF tahmin modelinin özellik seti oluyor. Analiz = feature engineering. |

---
**Veri kaynağı:** EPİAŞ Şeffaflık Platformu API v1 (`seffaflik.epias.com.tr`)
**Kimlik doğrulama:** CAS TGT ticket (`giris.epias.com.tr/cas/v1/tickets`)

## 0. Kurulum

`SYNTHETIC = True` bırakırsan notebook **kimlik bilgisi olmadan** sentetik ama
gerçekçi bir veriyle uçtan uca çalışır — grafiklerin ne göstereceğini görmek ve
kodu denemek için. Gerçek veri için `SYNTHETIC = False` yapıp aşağıya EPİAŞ
kullanıcı adı/şifreni gir.

In [ ]:
import os, re, json, time, warnings, urllib.request, urllib.parse, urllib.error
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
try:
    from IPython.display import display
except ImportError:                      # düz python olarak çalıştırılırsa
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

# ------------------------------------------------------------------ AYARLAR
#   "synthetic" → kimlik gerekmez, sentetik veriyle uçtan uca çalışır (deneme için)
#   "api"       → EPİAŞ Şeffaflık API'sinden canlı çeker (kullanıcı adı/şifre ister)
#   "csv"       → Şeffaflık sitesinden indirdiğin CSV/XLSX dosyalarını okur
MODE = "synthetic"

START_DATE  = "2022-01-01"
END_DATE    = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
CACHE_DIR   = Path("epias_cache"); CACHE_DIR.mkdir(exist_ok=True)
CSV_DIR     = Path("epias_csv");   CSV_DIR.mkdir(exist_ok=True)
TZ          = "Europe/Istanbul"   # Türkiye 2016'dan beri kalıcı UTC+3, DST yok

SYNTHETIC = (MODE == "synthetic")   # sonraki hücrelerle uyum için

# --- pandas sürüm uyumu (Colab'daki sürüm eski olabilir) --------------------
_PDV   = tuple(int(x) for x in pd.__version__.split(".")[:2])
FREQ_H = "h"  if _PDV >= (2, 2) else "H"     # saatlik
FREQ_Y = "YE" if _PDV >= (2, 2) else "A"     # yıl sonu

print(f"Analiz aralığı: {START_DATE} → {END_DATE}   ·   MODE = {MODE}")
print(f"pandas {pd.__version__} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")

# --------------------------------------------------- GÖRSEL STİL (tek yerden)
# Okabe-Ito paleti: renk körlüğüne karşı doğrulanmış, sabit sırayla atanır.
CAT = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7",
       "#56B4E9", "#F0E442", "#7F7F7F", "#332288"]
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#F2F7FB", "#0072B2", "#04304C"])
DIV = LinearSegmentedColormap.from_list("div_bo",  ["#0072B2", "#EDEDED", "#D55E00"])
INK, MUTED, GRID = "#1A1A1A", "#5C5C5C", "#DDDDDD"

plt.rcParams.update({
    "figure.figsize": (12, 4.2), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlepad": 12, "axes.labelsize": 10,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "grid.alpha": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9,
    "lines.linewidth": 1.8, "font.size": 10,
})

def finish(ax, title=None, sub=None, ylab=None, xlab=None, legend=True):
    """Her grafikte aynı bitirme işlemleri: başlık + alt açıklama + eksen etiketi."""
    if title: ax.set_title(title, pad=18 if sub else 12)
    if sub:   ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=9,
                      color=MUTED, va="bottom")
    if ylab:  ax.set_ylabel(ylab)
    if xlab:  ax.set_xlabel(xlab)
    if legend and ax.get_legend_handles_labels()[0]:
        ax.legend(loc="best", ncols=min(4, len(ax.get_legend_handles_labels()[0])))
    ax.set_axisbelow(True)
    plt.tight_layout()
    return ax

### 0.1 EPİAŞ API istemcisi

EPİAŞ'ın yeni (v1) servisleri **tüm uç noktalarda** kimlik doğrulama istiyor:

1. `POST https://giris.epias.com.tr/cas/v1/tickets` → gövdede `username=...&password=...`
   (form-urlencoded). Dönen düz metin `TGT-...` ile başlar. Ömrü ~2 saat.
2. Her veri çağrısı: `POST https://seffaflik.epias.com.tr/electricity-service/v1/...`
   başlıkta `TGT: <ticket>`, gövdede `{"startDate": "...", "endDate": "..."}`
   (ISO-8601, `+03:00` ofsetli). Cevap: `{"items": [...]}`.

> Hazır kütüphane tercih edersen `pip install eptr2` aynı işi yapıyor. Burada
> ne olup bittiğini görebilmek için ~60 satırlık kendi istemcimizi yazıyoruz.

In [ ]:
class Epias:
    ROOT  = "https://seffaflik.epias.com.tr"
    LOGIN = "https://giris.epias.com.tr/cas/v1/tickets"

    # Doğrulanmış uç nokta yolları
    PATHS = {
        "mcp":          "electricity-service/v1/markets/dam/data/mcp",                 # PTF
        "smp":          "electricity-service/v1/markets/bpm/data/system-marginal-price",# SMF
        "smp-dir":      "electricity-service/v1/markets/bpm/data/system-direction",     # Sistem yönü
        "supply-demand":"electricity-service/v1/markets/dam/data/supply-demand",        # GÖP arz-talep eğrisi
        "dam-volume":   "electricity-service/v1/markets/dam/data/day-ahead-market-trade-volume",
        "dam-clearing": "electricity-service/v1/markets/dam/data/clearing-quantity",
        "wap":          "electricity-service/v1/markets/idm/data/weighted-average-price",# GİP AOF
        "rt-cons":      "electricity-service/v1/consumption/data/realtime-consumption",  # Gerçek zamanlı tüketim
        "load-plan":    "electricity-service/v1/consumption/data/load-estimation-plan",  # Yük tahmin planı (YTP)
        "uecm":         "electricity-service/v1/consumption/data/uecm",
        "rt-gen":       "electricity-service/v1/generation/data/realtime-generation",    # Gerçek zamanlı üretim
        "kgup":         "electricity-service/v1/generation/data/dpp",                    # KGÜP
        "eak":          "electricity-service/v1/generation/data/aic",                    # Emre amade kapasite
        "wind-forecast":"electricity-service/v1/renewables/data/res-generation-and-forecast",
        "pp-list":      "electricity-service/v1/generation/data/powerplant-list",
    }
    # Bazı uç noktalar ek zorunlu parametre istiyor
    EXTRA_REQUIRED = {"kgup": {"region": "TR1"}, "eak": {"region": "TR1"}}

    def __init__(self, username, password, timeout=60):
        self.u, self.p, self.timeout = username, password, timeout
        self._tgt, self._tgt_at = None, None

    # ---------------------------------------------------------------- login
    def tgt(self):
        if self._tgt and (time.time() - self._tgt_at) < 90 * 60:
            return self._tgt
        body = urllib.parse.urlencode({"username": self.u, "password": self.p}).encode()
        req = urllib.request.Request(
            self.LOGIN, data=body, method="POST",
            headers={"Content-Type": "application/x-www-form-urlencoded",
                     "Accept": "text/plain"})
        with urllib.request.urlopen(req, timeout=self.timeout) as r:
            txt = r.read().decode("utf-8").strip()
        if not txt.startswith("TGT-"):
            raise RuntimeError(f"Giriş başarısız, TGT dönmedi: {txt[:200]}")
        self._tgt, self._tgt_at = txt, time.time()
        return self._tgt

    # ----------------------------------------------------------- tek çağrı
    @staticmethod
    def _iso(d):
        return pd.Timestamp(d).strftime("%Y-%m-%dT00:00:00+03:00")

    def call(self, key, **params):
        path = self.PATHS[key]
        body = dict(self.EXTRA_REQUIRED.get(key, {}))
        for k, v in params.items():
            body[k] = self._iso(v) if k in ("startDate", "endDate", "date", "period") else v
        req = urllib.request.Request(
            f"{self.ROOT}/{path}", data=json.dumps(body).encode(), method="POST",
            headers={"Content-Type": "application/json", "TGT": self.tgt()})
        for attempt in range(4):
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    return json.loads(r.read().decode("utf-8"))
            except urllib.error.HTTPError as e:
                msg = e.read().decode("utf-8", "ignore")[:300]
                if e.code in (401, 403):            # ticket düşmüş olabilir
                    self._tgt = None
                    req.headers["Tgt"] = self.tgt()
                elif e.code in (429, 500, 502, 503, 504):
                    time.sleep(2 ** attempt)
                else:
                    raise RuntimeError(f"{key} → HTTP {e.code}: {msg}") from None
            except urllib.error.URLError:
                time.sleep(2 ** attempt)
        raise RuntimeError(f"{key} çağrısı {4} denemede başarısız.")

    # --------------------------------------- tarih aralığını parçalayarak çek
    def fetch(self, key, start, end, chunk_months=3, sleep=0.25, verbose=True):
        """EPİAŞ uzun aralıklarda hata veriyor; aralığı parçalara bölüp birleştirir."""
        edges = pd.date_range(start, pd.Timestamp(end) + pd.Timedelta(days=1),
                              freq=f"{chunk_months}MS").tolist()
        if pd.Timestamp(start) not in edges: edges = [pd.Timestamp(start)] + edges
        last = pd.Timestamp(end) + pd.Timedelta(days=1)
        if edges[-1] < last: edges.append(last)
        frames = []
        for a, b in zip(edges[:-1], edges[1:]):
            res = self.call(key, startDate=a, endDate=b - pd.Timedelta(days=1))
            items = res.get("items") or res.get("body", {}).get("content", []) or []
            if items: frames.append(pd.DataFrame(items))
            if verbose:
                print(f"  {key}: {a.date()} → {(b - pd.Timedelta(days=1)).date()}  "
                      f"({len(items)} satır)")
            time.sleep(sleep)
        if not frames:
            return pd.DataFrame()
        return pd.concat(frames, ignore_index=True)

### 0.2 Cevapları saatlik tabloya normalize etme

EPİAŞ cevaplarında `date` alanı bazen tam ISO zaman damgası, bazen gün + ayrı
`hour`/`time` alanı olarak geliyor. Aşağıdaki yardımcı ikisini de kaldırıyor ve
**kolon adlarını olduğu gibi yazdırıyor** — API alan adları zaman zaman
değiştiği için körlemesine isim varsaymak yerine gerçeği görmek daha güvenli.

In [ ]:
def normalize(df, label="", show=True):
    """items listesinden gelen DataFrame'i saatlik, DatetimeIndex'li tabloya çevirir."""
    if df is None or len(df) == 0:
        print(f"[{label}] boş"); return pd.DataFrame()
    df = df.copy()
    if show:
        print(f"[{label}] ham kolonlar: {list(df.columns)}")

    tcol = next((c for c in ("date", "dateTime", "time", "tarih") if c in df.columns), None)
    if tcol is None:
        raise KeyError(f"[{label}] zaman kolonu bulunamadı: {list(df.columns)}")
    ts = pd.to_datetime(df[tcol], errors="coerce", utc=True).dt.tz_convert("Etc/GMT-3").dt.tz_localize(None)

    # 'date' sadece gün ise (hepsi 00:00) ve ayrı bir saat kolonu varsa ekle
    hcol = next((c for c in ("hour", "time", "saat") if c in df.columns and c != tcol), None)
    if hcol is not None and (ts.dt.hour.fillna(0) == 0).all():
        hh = df[hcol].astype(str).str.extract(r"(\d{1,2})")[0].astype(float)
        ts = ts + pd.to_timedelta(hh.fillna(0), unit="h")

    out = df.drop(columns=[c for c in (tcol, hcol) if c is not None], errors="ignore")
    out = out.apply(pd.to_numeric, errors="coerce")
    out = out.dropna(axis=1, how="all")
    out.index = pd.DatetimeIndex(ts, name="ts")
    out = out[~out.index.isna()]
    out = out[~out.index.duplicated(keep="last")].sort_index()
    return out


def cached(name, builder):
    """Aynı veriyi tekrar tekrar çekmemek için basit disk önbelleği."""
    f = CACHE_DIR / f"{name}.pkl"
    if f.exists():
        print(f"[cache] {name}")
        return pd.read_pickle(f)
    df = builder()
    df.to_pickle(f)
    return df

### 0.3 Yakıt grupları — merit order'ın iskeleti

Analizin geri kalanı bu ayrıma dayanıyor:

* **Sıfır marjinal maliyetli / must-run:** rüzgâr, güneş, jeotermal, biyokütle,
  akarsu (nehir tipi), atık ısı, nükleer. Yakıt maliyeti ~0 ya da üretimi
  fiyattan bağımsız. Merit order'da **en solda** yer alır, her koşulda üretir.
* **Esnek / marjinal olabilen:** doğalgaz, ithal kömür, linyit, taş kömürü,
  asfaltit, fuel-oil, motorin, LNG, nafta, **barajlı hidro** (su bir opsiyon
  değeri taşıdığı için fırsat maliyetiyle teklif verir).

**Artık yük (residual load) = Tüketim − (sıfır marjinal maliyetli üretim)**

Fiyatı belirleyen budur; brüt tüketim değil. 40 GW talebin 12 GW'ı rüzgârdan
geliyorsa piyasa 28 GW'lık bir talep görüyor demektir.

In [ ]:
ZERO_MC = ["wind", "sun", "geothermal", "biomass", "river", "wasteheat", "nuclear"]
FLEX    = ["naturalGas", "importCoal", "lignite", "blackCoal", "asphaltiteCoal",
           "fueloil", "gasOil", "lng", "naphta", "dammedHydro"]
# Renkler yakıt KİMLİĞİNE sabitlenmiştir (döngüsel atama yok): soğuk yeşil/sarı ailesi
# sıfır marjinal maliyetli üretim, mavi→gri→turuncu ailesi esnek/termik üretim.
FUEL_COLOR = {
    "wind": "#2E8B57", "sun": "#F0C419", "river": "#4FB3A5", "geothermal": "#8FBF3F",
    "biomass": "#5F8A2A", "wasteheat": "#A9CFC0", "nuclear": "#7E57C2",
    "dammedHydro": "#4A90C2", "lignite": "#6B4E2E", "importCoal": "#2B4A6F",
    "blackCoal": "#3E3E42", "asphaltiteCoal": "#9C7B57", "naturalGas": "#E07B39",
    "fueloil": "#B03A2E", "gasOil": "#D98880", "lng": "#F1948A", "naphta": "#7B241C",
}
fcol = lambda c: FUEL_COLOR.get(c, "#9E9E9E")

TR_NAME = {
    "wind": "Rüzgâr", "sun": "Güneş", "geothermal": "Jeotermal", "biomass": "Biyokütle",
    "river": "Akarsu", "wasteheat": "Atık ısı", "nuclear": "Nükleer",
    "naturalGas": "Doğalgaz", "importCoal": "İthal kömür", "lignite": "Linyit",
    "blackCoal": "Taş kömürü", "asphaltiteCoal": "Asfaltit", "fueloil": "Fuel-oil",
    "gasOil": "Motorin", "lng": "LNG", "naphta": "Nafta", "dammedHydro": "Barajlı hidro",
}

### 0.4 Veriyi çek

Çekilen seriler:

| Anahtar | Seri | Rolü |
|---|---|---|
| `rt-cons` | Gerçek zamanlı tüketim | **Talep** — yük eğrisinin kendisi |
| `load-plan` | Yük tahmin planı (YTP) | Piyasanın *ex-ante* talep beklentisi |
| `mcp` | PTF | **Fiyat** — açıklamaya çalıştığımız değişken |
| `smp` | SMF | Dengeleme piyasası fiyatı → sistemin gerçek zamanlı sıkışıklığı |
| `rt-gen` | Gerçek zamanlı üretim (yakıt bazlı) | **Arz** — merit order'ın gözlenen hâli |
| `eak` | Emre amade kapasite | Arz tavanı → yedek marj |
| `kgup` | KGÜP | Üreticilerin *ex-ante* programı (tahminlemede sızıntısız girdi) |

In [ ]:
def make_synthetic(start, end, seed=7):
    """Gerçek API şemasıyla aynı kolon adlarını üreten sentetik veri."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, pd.Timestamp(end) + pd.Timedelta(hours=23), freq=FREQ_H)
    n = len(idx)
    doy, hod, dow = idx.dayofyear.values, idx.hour.values, idx.dayofweek.values
    yr = idx.year.values - 2022

    def ar1(rho, sd, lo, hi):
        e = rng.normal(0, sd, n); x = np.zeros(n)
        for i in range(1, n): x[i] = rho * x[i-1] + e[i]
        return np.clip(x, lo, hi)

    # --- Talep: yıllık büyüme + kış/yaz çift zirvesi + hafta ritmi + çift tepe profil
    base   = 33000 * (1 + 0.022 * yr)
    season = 4200*np.cos(2*np.pi*(doy-12)/365) + 3800*np.clip(np.cos(2*np.pi*(doy-205)/365), 0, 1)**1.4
    week   = np.where(dow == 6, -0.085, np.where(dow == 5, -0.040, 0.012))
    # Gerçekçi Türkiye günlük profili: gece dibi ~05:00, gündüz platosu, akşam zirvesi
    HPROF = np.array([-.10,-.14,-.17,-.19,-.20,-.19,-.15,-.07, .02, .08, .12, .13,
                      .12, .10, .09, .09, .10, .12, .13, .15, .14, .10, .04,-.03])
    w_yaz  = np.clip(np.cos(2*np.pi*(doy-205)/365), 0, 1)     # yaz ağırlığı
    w_kis  = np.clip(np.cos(2*np.pi*(doy-15)/365),  0, 1)     # kış ağırlığı
    daily  = (HPROF[hod]
              + 0.055*w_yaz*np.exp(-((hod-15)**2)/9)          # yazın öğleden sonra klima zirvesi
              + 0.055*w_kis*np.exp(-((hod-19)**2)/6))         # kışın erken akşam zirvesi
    cons = (base + season) * (1 + week + daily) + ar1(0.93, 260, -4000, 4000)

    # --- Sıfır marjinal maliyetli üretim
    wind = np.clip(3600 + 2600*np.cos(2*np.pi*(doy-20)/365) + ar1(0.965, 620, -6000, 8000), 120, 11500)
    sun_cap = 5200 + 3400*yr + 1200*np.sin(2*np.pi*doy/365)   # TR'de güneş kurulu gücü hızla büyüdü
    sun = np.clip(sun_cap*np.exp(-((hod-13.0)**2)/9.0)*(0.72+0.28*np.cos(2*np.pi*(doy-172)/365))
                  * rng.uniform(0.55, 1.0, n), 0, None)
    sun[(hod < 6) | (hod > 19)] = 0.0
    river = np.clip(3900 + 1900*np.cos(2*np.pi*(doy-105)/365) + ar1(0.98, 190, -1800, 2400), 700, None)
    geothermal = 1450 + 40*yr + rng.normal(0, 45, n)
    biomass    = 900 + 190*yr + rng.normal(0, 40, n)
    wasteheat  = 260 + rng.normal(0, 18, n)
    nuclear    = np.where(idx.year.values >= 2026, 1100.0, 0.0)

    renew = wind + sun + river + geothermal + biomass + wasteheat + nuclear
    resid = cons - renew

    # --- Esnek üretim: barajlı hidro zirveyi yontar, gaz artakalanı kapatır
    rq = pd.Series(resid).rank(pct=True).values
    dammedHydro = np.clip(1700 + 6200*rq**2.1 + rng.normal(0, 320, n), 250, None)
    importCoal  = np.clip(7600 + 900*rq + rng.normal(0, 260, n), 0, None)
    lignite     = np.clip(5900 + 1500*rq + rng.normal(0, 300, n), 0, None)
    blackCoal   = np.clip(1500 + rng.normal(0, 110, n), 0, None)
    naturalGas  = np.clip(resid - dammedHydro - importCoal - lignite - blackCoal, 250, None)
    # --- Fiyat: merit order eğrisi (dışbükey) × yakıt maliyeti endeksi + kıtlık primi
    gas_ix = np.interp(idx.values.astype("datetime64[D]").astype(float),
                       [pd.Timestamp(d).to_pydatetime().toordinal()-719163 for d in
                        ["2022-01-01","2022-09-01","2023-06-01","2024-06-01","2025-06-01","2027-01-01"]],
                       [1.00, 1.85, 1.05, 1.12, 1.22, 1.30])
    rnorm = (resid - np.percentile(resid, 1)) / (np.percentile(resid, 99) - np.percentile(resid, 1))
    rnorm = np.clip(rnorm, 0, 1.25)
    mc = 620 + 2450*rnorm**3.1                      # dışbükey merit order
    outage    = np.clip(ar1(0.995, 260, -4000, 9000) + 2600, 0, None)   # planlı/plansız bakım
    eak_total = 55000 + 1500*yr - outage + rng.normal(0, 500, n)
    margin = (eak_total - resid) / eak_total
    scarcity = 1 + 2.6*np.clip(0.16 - margin, 0, None)/0.16
    # YEK'in artık yük dışındaki ek etkisi (teklif davranışı / YEKDEM): merit order etkisi
    vre_extra = np.exp(-0.011 * (wind + sun)/1000)
    ptf = mc * gas_ix * scarcity * vre_extra * np.exp(rng.normal(0, 0.085, n))
    cap = np.where(idx.year.values <= 2022, 4800, np.where(idx.year.values <= 2023, 4200, 3400.0))
    ptf = np.clip(ptf, 0, cap)
    ptf[rng.random(n) < 0.004] = 0.0                # nadir sıfır fiyat saatleri
    smf = np.clip(ptf * np.exp(rng.normal(0.02, 0.22, n)), 0, cap * 1.05)
    lep = cons * (1 + rng.normal(0, 0.021, n))      # YTP: gerçekleşene yakın ama hatalı

    gen = pd.DataFrame(
        {"wind": wind, "sun": sun, "river": river, "geothermal": geothermal,
         "biomass": biomass, "wasteheat": wasteheat, "nuclear": nuclear,
         "naturalGas": naturalGas, "importCoal": importCoal, "lignite": lignite,
         "blackCoal": blackCoal, "dammedHydro": dammedHydro,
         "fueloil": np.clip(rng.normal(55, 20, n), 0, None),
         "asphaltiteCoal": np.clip(rng.normal(370, 60, n), 0, None)}, index=idx)
    gen["total"] = gen.sum(axis=1)

    return {
        "rt-cons":   pd.DataFrame({"consumption": cons}, index=idx),
        "load-plan": pd.DataFrame({"lep": lep}, index=idx),
        "mcp":       pd.DataFrame({"price": ptf, "priceUsd": ptf/np.linspace(15, 41, n)}, index=idx),
        "smp":       pd.DataFrame({"systemMarginalPrice": smf}, index=idx),
        "rt-gen":    gen,
        "eak":       pd.DataFrame({"toplam": eak_total}, index=idx),
    }

### 0.4b CSV yolu — Şeffaflık sitesinden indirilen dosyaları okumak

API çalışmazsa (hesabında "Şeffaflık Servisleri" yetkisi kapalı olabilir) aynı
veriyi siteden indirip yükleyebilirsin. Aşağıdaki okuyucu EPİAŞ'ın Türkçe CSV/XLSX
çıktısını olduğu gibi kabul ediyor: `01.01.2022` tarih biçimi, `1.234,56` sayı
biçimi ve Türkçe kolon başlıkları otomatik çevriliyor.

**İndirilecek 6 seri** (her biri Şeffaflık'ta "Dışa Aktar" düğmesiyle):

| # | Menü yolu | Dosya adına şunu koy |
|---|---|---|
| 1 | Tüketim → Gerçek Zamanlı Tüketim | `tuketim` |
| 2 | Tüketim → Yük Tahmin Planı | `ytp` |
| 3 | Piyasalar → Gün Öncesi Piyasası → Piyasa Takas Fiyatı | `ptf` |
| 4 | Piyasalar → Dengeleme Güç Piyasası → Sistem Marjinal Fiyatı | `smf` |
| 5 | Üretim → Gerçek Zamanlı Üretim | `uretim` |
| 6 | Üretim → Emre Amade Kapasite | `eak` |

*(İsteğe bağlı: Üretim → KGÜP, dosya adında `kgup`. Tahmin bölümü için gerekli.)*

Site tek seferde sınırlı aralık indirtiyor; 2022–bugün için yıl yıl indirip
hepsini birden yüklemen yeterli — okuyucu aynı seriye ait parçaları birleştiriyor.

Colab'da dosyaları yüklemek için bir sonraki hücreyi çalıştır.

In [ ]:
TR2EN = {   # EPİAŞ Türkçe yakıt başlıkları → notebook'un iç adları
    "doğal gaz": "naturalGas", "doğalgaz": "naturalGas",
    "barajlı": "dammedHydro", "barajlı hidro": "dammedHydro", "baraj": "dammedHydro",
    "linyit": "lignite", "akarsu": "river", "nehir": "river",
    "ithal kömür": "importCoal", "ithalkömür": "importCoal",
    "rüzgar": "wind", "rüzgâr": "wind", "güneş": "sun", "gunes": "sun",
    "fuel oil": "fueloil", "fueloil": "fueloil", "fuel-oil": "fueloil",
    "jeo termal": "geothermal", "jeotermal": "geothermal",
    "asfaltit kömür": "asphaltiteCoal", "asfaltit": "asphaltiteCoal",
    "taş kömür": "blackCoal", "taş kömürü": "blackCoal", "taşkömürü": "blackCoal",
    "biyokütle": "biomass", "nafta": "naphta", "lng": "lng", "motorin": "gasOil",
    "atık isı": "wasteheat", "atık ısı": "wasteheat", "atikisi": "wasteheat",
    "nükleer": "nuclear", "toplam": "total",
    "uluslararası": "importExport", "ithalat/ihracat": "importExport",
}
FILE_HINTS = [   # dosya adında geçen ipucu → seri anahtarı  (sıra önemli)
    ("kgup", "kgup"), ("kgüp", "kgup"), ("dpp", "kgup"),
    ("eak", "eak"), ("emre", "eak"),
    ("ptf", "mcp"), ("mcp", "mcp"), ("takas", "mcp"),
    ("smf", "smp"), ("marjinal", "smp"),
    ("ytp", "load-plan"), ("yuk", "load-plan"), ("yük", "load-plan"), ("load", "load-plan"),
    ("tuketim", "rt-cons"), ("tüketim", "rt-cons"), ("consumption", "rt-cons"),
    ("uretim", "rt-gen"), ("üretim", "rt-gen"), ("generation", "rt-gen"),
]

def _tr_number(sr):
    """'1.234,56' → 1234.56 ; zaten sayıysa dokunma."""
    if pd.api.types.is_numeric_dtype(sr):
        return sr
    return pd.to_numeric(
        sr.astype(str).str.replace(r"[^\d,.\-]", "", regex=True)
                      .str.replace(".", "", regex=False)
                      .str.replace(",", ".", regex=False)
                      .replace({"": None, "-": None}),
        errors="coerce")

def read_epias_file(path):
    """Bir EPİAŞ CSV/XLSX dosyasını saatlik, DatetimeIndex'li tabloya çevirir."""
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:
        for enc in ("utf-8-sig", "utf-8", "cp1254", "iso-8859-9"):
            try:
                df = pd.read_csv(path, sep=None, engine="python", encoding=enc)
                break
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
        else:
            raise IOError(f"{path.name} okunamadı")

    df.columns = [str(c).strip() for c in df.columns]
    low = {c: c.lower() for c in df.columns}

    tcol = next((c for c in df.columns if low[c].startswith(("tarih", "date"))), None)
    hcol = next((c for c in df.columns if low[c].startswith(("saat", "hour", "time"))), None)
    if tcol is None:
        raise KeyError(f"{path.name}: 'Tarih' kolonu yok → {list(df.columns)}")

    ts = pd.to_datetime(df[tcol], dayfirst=True, errors="coerce")
    if hcol is not None:
        hh = df[hcol].astype(str).str.extract(r"(\d{1,2})")[0].astype(float)
        ts = ts.dt.normalize() + pd.to_timedelta(hh.fillna(0), unit="h")

    out = pd.DataFrame(index=pd.DatetimeIndex(ts, name="ts"))
    for c in df.columns:
        if c in (tcol, hcol):
            continue
        base = re.sub(r"\(.*?\)", "", c).strip().lower()      # "PTF (TL/MWh)" → "ptf"
        name = TR2EN.get(base, base)
        cl = c.lower()                                        # birimi koru:
        if   "usd" in cl: name += "Usd"                       # aynı seri TL/USD/EUR
        elif "eur" in cl: name += "Eur"                       # olarak üç kez gelebilir
        out[name] = _tr_number(df[c]).values
    out = out.dropna(axis=1, how="all")
    return out[~out.index.isna()].sort_index()

def guess_key(path, df):
    """Dosya adından, olmazsa kolonlarından hangi seri olduğunu bul."""
    n = Path(path).name.lower()
    for hint, key in FILE_HINTS:
        if hint in n:
            return key
    cols = set(df.columns)
    if len(cols & set(ZERO_MC + FLEX)) >= 4:
        return "rt-gen"
    for cand, key in [("ptf", "mcp"), ("price", "mcp"), ("smf", "smp"),
                      ("tüketim", "rt-cons"), ("consumption", "rt-cons"),
                      ("yük tahmin planı", "load-plan"), ("ytp", "load-plan")]:
        if any(cand in c.lower() for c in cols):
            return key
    return None

def load_from_csv(folder=CSV_DIR):
    """Klasördeki bütün EPİAŞ dosyalarını okur, seriye göre gruplar ve birleştirir."""
    files = sorted([p for p in Path(folder).iterdir()
                    if p.suffix.lower() in (".csv", ".xlsx", ".xls")])
    if not files:
        raise FileNotFoundError(
            f"'{folder}' klasöründe dosya yok. Bir üstteki hücreyle dosyaları yükle.")
    buckets = {}
    for f in files:
        df = read_epias_file(f)
        key = guess_key(f, df)
        if key is None:
            print(f"  ? {f.name}: hangi seri olduğu anlaşılamadı, atlandı "
                  f"(dosya adına ptf/smf/tuketim/ytp/uretim/eak ekle)")
            continue
        buckets.setdefault(key, []).append(df)
        print(f"  ✓ {f.name}  →  {key}  ({len(df)} satır, {len(df.columns)} kolon)")

    raw = {}
    for k, parts in buckets.items():
        d = pd.concat(parts).sort_index()
        raw[k] = d[~d.index.duplicated(keep="last")]
    eksik = [k for k in ("mcp", "smp", "rt-cons", "load-plan", "rt-gen") if k not in raw]
    if eksik:
        print(f"\n  ! Eksik seriler: {eksik} — bunlar olmadan bazı bölümler çalışmaz.")
    return raw

**Colab'da dosya yüklemek için** aşağıdaki hücreyi `MODE = "csv"` iken çalıştır.
Çalıştırınca bir "Dosya Seç" düğmesi çıkar; indirdiğin bütün EPİAŞ dosyalarını
birden seçebilirsin.

In [ ]:
if MODE == "csv":
    try:
        from google.colab import files as _colab_files
        print("İndirdiğin EPİAŞ dosyalarını seç (hepsini birden seçebilirsin):")
        up = _colab_files.upload()
        for fname in up:
            os.replace(fname, CSV_DIR / fname)
        print(f"\n{len(up)} dosya '{CSV_DIR}' klasörüne taşındı.")
    except ImportError:
        print(f"Colab dışındasın — dosyaları elle '{CSV_DIR}' klasörüne koy.")
    print("Klasördeki dosyalar:", [p.name for p in CSV_DIR.iterdir()])

In [ ]:
if MODE == "synthetic":
    raw = make_synthetic(START_DATE, END_DATE)
    print("Sentetik veri üretildi:", {k: v.shape for k, v in raw.items()})

elif MODE == "csv":
    print("CSV dosyaları okunuyor...")
    raw = load_from_csv()
    print("\nOkunan seriler:", {k: v.shape for k, v in raw.items()})

elif MODE == "api":
    from getpass import getpass
    EPIAS_USER = os.environ.get("EPIAS_USER") or input("EPİAŞ kullanıcı adı (e-posta): ")
    EPIAS_PASS = os.environ.get("EPIAS_PASS") or getpass("EPİAŞ şifre: ")
    ep = Epias(EPIAS_USER, EPIAS_PASS)
    print("TGT alındı:", ep.tgt()[:14], "...\n")

    raw = {}
    for k in ["rt-cons", "load-plan", "mcp", "smp", "rt-gen", "eak"]:
        raw[k] = cached(
            f"{k}_{START_DATE}_{END_DATE}",
            lambda k=k: normalize(ep.fetch(k, START_DATE, END_DATE), label=k))
        print(f"{k}: {raw[k].shape}\n")
else:
    raise ValueError(f"MODE 'synthetic', 'api' veya 'csv' olmalı — '{MODE}' verildi.")

### 0.5 Ana tablo (`D`) — tek saatlik iskelet

Bütün seriler tek bir saatlik tabloda birleşiyor. Türetilen kolonlar:

* `renew` — sıfır marjinal maliyetli üretim toplamı
* `residual` — **artık yük** = `consumption − renew`
* `flex` — esnek (marjinal olabilen) üretim toplamı
* `ren_share` — yenilenebilirin talebe oranı
* `lep_err` — YTP tahmin hatası (gerçekleşen − plan)

In [ ]:
def build_master(raw):
    D = pd.DataFrame(index=raw["mcp"].index)

    def pick(df, cands, new):
        c = next((c for c in cands if c in df.columns), None)
        if c is None:
            c = df.select_dtypes("number").columns[0]
            print(f"  ! {new}: beklenen kolon yok, '{c}' kullanıldı ({list(df.columns)})")
        D[new] = df[c]

    pick(raw["mcp"],       ["price", "mcp", "priceTl", "ptf",
                            "piyasa takas fiyatı"],                        "ptf")
    _usd = next((c for c in ("priceUsd", "ptfUsd", "piyasa takas fiyatıUsd")
             if c in raw["mcp"].columns), None)
    if _usd: D["ptf_usd"] = raw["mcp"][_usd]
    pick(raw["smp"],       ["systemMarginalPrice", "price", "smp", "smf",
                            "sistem marjinal fiyat"],                      "smf")
    pick(raw["rt-cons"],   ["consumption", "consumptionQuantity", "value",
                            "tüketim", "tuketim"],                         "consumption")
    pick(raw["load-plan"], ["lep", "loadEstimationPlan", "value", "ytp",
                            "yük tahmin planı", "yuk tahmin plani"],       "lep")

    gen = raw["rt-gen"]
    for c in ZERO_MC + FLEX:
        if c in gen.columns:
            D[c] = gen[c]
    if "total" in gen.columns: D["gen_total"] = gen["total"]

    zc = [c for c in ZERO_MC if c in D.columns]
    fc = [c for c in FLEX    if c in D.columns]
    D["renew"]     = D[zc].sum(axis=1)
    D["flex"]      = D[fc].sum(axis=1)
    D["residual"]  = D["consumption"] - D["renew"]
    D["ren_share"] = D["renew"] / D["consumption"]
    D["vre"]       = D[[c for c in ("wind", "sun") if c in D.columns]].sum(axis=1)
    D["lep_err"]   = D["consumption"] - D["lep"]
    D["spread"]    = D["smf"] - D["ptf"]

    if "eak" in raw and len(raw["eak"]):
        eak = raw["eak"]
        _tot = next((c for c in ("toplam", "total") if c in eak.columns), None)
        D["eak_total"] = eak[_tot] if _tot else eak.select_dtypes("number").sum(axis=1)
        D["margin"]    = (D["eak_total"] - D["residual"]) / D["eak_total"]

    # Takvim kolonları
    D["hour"], D["dow"] = D.index.hour, D.index.dayofweek
    D["month"], D["year"] = D.index.month, D.index.year
    D["date"]     = D.index.normalize()
    D["is_wknd"]  = D["dow"] >= 5
    D["season"]   = pd.Categorical(
        D["month"].map({12:"Kış",1:"Kış",2:"Kış",3:"İlkbahar",4:"İlkbahar",5:"İlkbahar",
                        6:"Yaz",7:"Yaz",8:"Yaz",9:"Sonbahar",10:"Sonbahar",11:"Sonbahar"}),
        categories=["Kış","İlkbahar","Yaz","Sonbahar"], ordered=True)
    return D.sort_index()

D = build_master(raw)
print(f"\nAna tablo: {D.shape[0]:,} saat × {D.shape[1]} kolon   "
      f"({D.index.min():%Y-%m-%d} → {D.index.max():%Y-%m-%d})")
print("Eksik saat sayısı:",
      len(pd.date_range(D.index.min(), D.index.max(), freq=FREQ_H)) - len(D))
D[["ptf","smf","consumption","renew","residual","ren_share"]].describe().T.round(1)

---
# 1. Yük Eğrileri — Talep Tarafının Anatomisi

Elektrik talebi kısa vadede fiyata neredeyse hiç tepki vermez (fiyat-inelastik).
Bu yüzden fiyat denkleminde talep **dışsal bir girdi** gibi davranır. Önce onun
ritmini çıkarıyoruz: hangi saatte, hangi günde, hangi mevsimde ne kadar yük var?

Aradığımız yapılar:
* **Çift tepe** (sabah ~10-12, akşam ~19-21) — sanayi + konut bindirmesi
* **Kış zirvesi vs yaz zirvesi** — ısıtma vs soğutma; Türkiye'de ikisi de var
* **Hafta sonu düşüşü** — sanayi payının göstergesi
* **Baz yük / pik yük oranı** — sistem esnekliğine ne kadar ihtiyaç var

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.4))
ax.plot(D.index, D["consumption"]/1000, lw=0.35, color=CAT[0], alpha=0.30, label="Saatlik")
ax.plot(D.index, (D["consumption"].rolling(24*7, center=True).mean())/1000,
        lw=2.2, color=CAT[0], label="7 günlük ort.")
ax.plot(D.index, (D["consumption"].rolling(24*30, center=True).mean())/1000,
        lw=2.0, color=CAT[3], label="30 günlük ort.")
finish(ax, "1.1 · Saatlik elektrik tüketimi",
       "Yaz ve kış çift zirvesi; ilkbahar/sonbahar geçiş dönemlerinde dip",
       "GW")

**Ne okuyoruz:** Türkiye'de yükün *iki* mevsimsel zirvesi var — kış (ısıtma +
aydınlatma) ve yaz (soğutma). Aradaki geçiş dönemleri (nisan-mayıs, ekim) sistemin
en gevşek olduğu, dolayısıyla fiyatın en düşük olmasını beklediğimiz dönemler.
Bu, ileride fiyat eğrisinde aynı desenin çıkıp çıkmadığını kontrol edeceğimiz
ilk hipotez.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

prof = D.groupby(["season", "hour"], observed=True)["consumption"].mean().unstack(0)/1000
for i, s in enumerate(prof.columns):
    axes[0].plot(prof.index, prof[s], color=CAT[i], label=str(s), marker="o", ms=3)
finish(axes[0], "1.2 · Ortalama günlük yük profili (mevsime göre)",
       None, "GW", "Saat")
axes[0].set_xticks(range(0, 24, 3))

wk = D.groupby(["is_wknd", "hour"], observed=True)["consumption"].mean().unstack(0)/1000
axes[1].plot(wk.index, wk[False], color=CAT[0], label="Hafta içi", marker="o", ms=3)
axes[1].plot(wk.index, wk[True],  color=CAT[1], label="Hafta sonu", marker="o", ms=3)
axes[1].fill_between(wk.index, wk[True], wk[False], color=CAT[0], alpha=0.10)
finish(axes[1], "1.2b · Hafta içi vs hafta sonu",
       f"Ortalama fark: {(wk[False]-wk[True]).mean():.1f} GW "
       f"({100*(wk[False]-wk[True]).mean()/wk[False].mean():.1f}%)", "GW", "Saat")
axes[1].set_xticks(range(0, 24, 3))
plt.tight_layout()

Hafta sonu düşüşünün büyüklüğü **sanayi payının** doğrudan göstergesi. Bu fark
saat bazında sabit değilse (örneğin gündüz büyük, gece küçük) sanayi yükünün
vardiya yapısını görüyorsunuz demektir. Tahmin modelinde `hafta_sonu × saat`
etkileşim terimi bu yüzden neredeyse her zaman kazandırır.

In [ ]:
years = sorted(D["year"].unique())
fig, axes = plt.subplots(len(years), 1, figsize=(13, 1.75*len(years)), sharex=True)
axes = np.atleast_1d(axes)
vmin, vmax = np.percentile(D["consumption"]/1000, [1, 99])
for ax, y in zip(axes, years):
    sub = D[D["year"] == y]
    m = sub.pivot_table(index="hour", columns=sub.index.dayofyear,
                        values="consumption", aggfunc="mean")/1000
    im = ax.imshow(m.values, aspect="auto", origin="lower", cmap=SEQ,
                   vmin=vmin, vmax=vmax, extent=[1, 366, 0, 24])
    ax.set_ylabel(f"{y}\nsaat", fontsize=9); ax.set_yticks([0, 6, 12, 18, 24]); ax.grid(False)
axes[-1].set_xlabel("Yılın günü")
axes[0].set_title("1.3 · Yük ısı haritası — saat × gün", loc="left", pad=10)
fig.colorbar(im, ax=axes.tolist(), label="GW", pad=0.01, fraction=0.02)

Isı haritası tek grafikte üç şeyi gösteriyor: (1) mevsimsel zarf, (2) günlük
profilin mevsime göre **şekil değiştirmesi** — yazın zirve öğleden sonraya
kayar (klima), kışın akşam erken saatlere (aydınlatma+ısıtma); (3) tatil
etkileri (bayramlar, yılbaşı) dikey soluk şeritler olarak.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

for i, y in enumerate(years):
    v = np.sort(D.loc[D["year"] == y, "consumption"].values)[::-1]/1000
    axes[0].plot(np.linspace(0, 100, len(v)), v, color=CAT[i % len(CAT)], label=str(y))
finish(axes[0], "1.4 · Yük süre eğrisi (LDC)",
       "Yılın % kaçında yük şu seviyenin üstünde?", "GW", "Yılın yüzdesi (%)")

last = years[-2] if len(years) > 1 else years[-1]
v = np.sort(D.loc[D["year"] == last, "consumption"].values)[::-1]/1000
x = np.linspace(0, 100, len(v))
axes[1].plot(x, v, color=INK, lw=2)
thr = np.percentile(v, 85)          # yılın yalnızca %15'inde aşılan seviye
axes[1].fill_between(x, 0, v.min(), color=CAT[2], alpha=.30, label="Baz yük")
axes[1].fill_between(x, v.min(), np.minimum(v, thr), color=CAT[0], alpha=.25,
                     label="Orta yük")
axes[1].fill_between(x, np.minimum(v, thr), v, color=CAT[3], alpha=.35,
                     label="Pik yük (yılın %15'i)")
axes[1].plot(x, v, color=INK, lw=2)
axes[1].set_ylim(0, v.max()*1.06)
finish(axes[1], f"1.4b · {last} — baz / orta / pik ayrımı",
       f"Pik {v.max():.1f} GW · Baz {v.min():.1f} GW · Yük faktörü "
       f"{v.mean()/v.max():.2f}", "GW", "Yılın yüzdesi (%)")
plt.tight_layout()

**LDC neden kritik?** Merit order'ın *talep tarafındaki* aynası. Sol uçtaki dar
şerit (yılın ~%1-2'si) yılda birkaç yüz saat çalışan pahalı pik santralleriyle
karşılanır — fiyat spike'larının doğduğu yer tam burası. Sağdaki geniş taban ise
baz yük santralleriyle. Yük faktörü (`ortalama/pik`) ne kadar düşükse sistem o
kadar çok esnek kapasite taşımak zorunda, fiyat da o kadar volatil olur.

İleride aynı eğriyi **fiyat** için de çizeceğiz (fiyat süre eğrisi) ve ikisinin
şeklini karşılaştıracağız.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))
err = D["lep_err"]
mape = (err.abs() / D["consumption"]).groupby(D["year"]).mean() * 100

axes[0].bar(mape.index.astype(str), mape.values, color=CAT[0], width=.6)
for xi, vi in zip(range(len(mape)), mape.values):
    axes[0].text(xi, vi, f"{vi:.2f}%", ha="center", va="bottom", fontsize=9, color=INK)
finish(axes[0], "1.5 · YTP tahmin hatası (MAPE)",
       "Yük tahmin planı ile gerçekleşen tüketim arası sapma", "%", "Yıl", legend=False)

hp = err.groupby(D["hour"]).agg(["mean", lambda s: s.quantile(.1), lambda s: s.quantile(.9)])
hp.columns = ["mean", "p10", "p90"]
axes[1].fill_between(hp.index, hp["p10"]/1000, hp["p90"]/1000, color=CAT[0], alpha=.18, label="P10–P90")
axes[1].plot(hp.index, hp["mean"]/1000, color=CAT[0], marker="o", ms=3, label="Ortalama")
axes[1].axhline(0, color=MUTED, lw=1, ls="--")
finish(axes[1], "1.5b · Hata saate göre yanlı mı?",
       "Sistematik sapma varsa fiyat tahmininde düzeltilebilir bilgi vardır", "GW", "Saat")
axes[1].set_xticks(range(0, 24, 3))
plt.tight_layout()

**Bu grafik neden burada?** YTP, gün öncesi piyasa kapanmadan **önce** yayınlanır.
Yani PTF tahmininde kullanabileceğin, sızıntısız bir talep göstergesidir. İki şey
öğreniyoruz: (1) tahmin hatası ne kadar (fiyat tahmininin talep kaynaklı hata
tabanı), (2) hata saate/mevsime göre **yanlı** mı — yanlıysa model bunu öğrenip
düzeltebilir.

---
# 2. Fiyat Eğrileri — PTF'nin Anatomisi

Şimdi aynı soruları fiyata soruyoruz. Kritik nokta: **yükle aynı grafik dilini
kullanmak.** Aynı profil, aynı süre eğrisi, aynı ısı haritası. Şekiller
örtüşüyorsa fiyat talebi izliyor; örtüşmüyorsa arada bir şey var — o "bir şey"
arz tarafı olacak (Bölüm 3).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.4))
ax.plot(D.index, D["ptf"], lw=0.3, color=CAT[3], alpha=.30, label="Saatlik PTF")
ax.plot(D.index, D["ptf"].rolling(24*7, center=True).mean(), lw=2.2, color=CAT[3],
        label="7 günlük ort.")
ax.plot(D.index, D["ptf"].rolling(24*30, center=True).mean(), lw=2.0, color=INK,
        label="30 günlük ort.")
finish(ax, "2.1 · PTF zaman serisi", "Nominal TL/MWh — enflasyon ve tavan fiyat "
       "rejimleri seviyeyi kaydırır", "TL/MWh")

> **Uyarı — rejim kırılmaları.** PTF nominal TL cinsinden. 2022 enerji krizi,
> değişen **tavan fiyat** uygulamaları ve TL'deki değer kaybı seviyeyi
> kaydırıyor. Modellemede ham TL yerine ya (a) USD/EUR cinsinden PTF, ya
> (b) doğalgaz maliyet endeksine oranlanmış PTF, ya da (c) rejim kukla
> değişkenleri kullanmak gerekir. Aksi hâlde model "trend"i öğrenip arz-talep
> sinyalini kaçırır.

In [ ]:
if "ptf_usd" in D.columns:
    fig, ax = plt.subplots(figsize=(13, 3.8))
    ax.plot(D.index, D["ptf_usd"].rolling(24*30, center=True).mean(),
            color=CAT[2], lw=2.2, label="PTF (USD/MWh, 30g ort.)")
    finish(ax, "2.1b · Aynı seri USD cinsinden",
           "TL serisindeki trendin ne kadarı gerçek fiyat hareketi, ne kadarı kur?",
           "USD/MWh")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

pp = D.groupby(["season", "hour"], observed=True)["ptf"].mean().unstack(0)
for i, s in enumerate(pp.columns):
    axes[0].plot(pp.index, pp[s], color=CAT[i], label=str(s), marker="o", ms=3)
finish(axes[0], "2.2 · Ortalama saatlik PTF profili", None, "TL/MWh", "Saat")
axes[0].set_xticks(range(0, 24, 3))

# Yük ve fiyat profilini aynı ölçeğe indirip (z-skor) üst üste koy — çift eksen YOK
z = lambda s: (s - s.mean()) / s.std()
hp_c = z(D.groupby("hour")["consumption"].mean())
hp_r = z(D.groupby("hour")["residual"].mean())
hp_p = z(D.groupby("hour")["ptf"].mean())
axes[1].plot(hp_c.index, hp_c.values, color=CAT[0], marker="o", ms=3, label="Tüketim")
axes[1].plot(hp_r.index, hp_r.values, color=CAT[2], marker="s", ms=3, label="Artık yük")
axes[1].plot(hp_p.index, hp_p.values, color=CAT[3], marker="^", ms=3, label="PTF")
axes[1].axhline(0, color=MUTED, lw=1, ls="--")
finish(axes[1], "2.2b · Yük mü, artık yük mü fiyatın şeklini tutuyor?",
       "Üçü de z-skora normalize edildi (tek eksen)", "z-skor", "Saat")
axes[1].set_xticks(range(0, 24, 3))
plt.tight_layout()

print("Saatlik profil korelasyonu (PTF ~ ...) — yaz aylarında, yıl bazında")
print(f"{'Yıl':<6}{'Tüketim':>10}{'Artık yük':>12}")
for y in years:
    d = D[(D["year"] == y) & (D["season"] == "Yaz")]
    if len(d) < 500: continue
    q = d.groupby("hour")[["consumption", "residual", "ptf"]].mean()
    print(f"{y:<6}{q['consumption'].corr(q['ptf']):>10.3f}{q['residual'].corr(q['ptf']):>12.3f}")

In [ ]:
# Ördek eğrisinin yıllar içindeki evrimi — yaz ayları
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for i, y in enumerate(years):
    d = D[(D["year"] == y) & (D["season"] == "Yaz")]
    if len(d) < 500: continue
    axes[0].plot(range(24), d.groupby("hour")["residual"].mean()/1000,
                 color=CAT[i % len(CAT)], marker="o", ms=3, label=str(y))
    axes[1].plot(range(24), d.groupby("hour")["ptf"].mean()/d["ptf"].mean(),
                 color=CAT[i % len(CAT)], marker="o", ms=3, label=str(y))
finish(axes[0], "2.2c · Ördek eğrisi: yaz artık yük profili",
       "Güneş kapasitesi büyüdükçe öğle çukuru derinleşir", "GW", "Saat")
axes[1].axhline(1, color=MUTED, ls="--", lw=1)
finish(axes[1], "2.2d · Aynı yazların PTF profili (yıl ortalamasına oranla)",
       "Fiyat çukuru artık yük çukurunu takip ediyor mu?", "PTF / yıl ort.", "Saat")
for a in axes: a.set_xticks(range(0, 24, 3))
plt.tight_layout()

**İlk büyük bulgu burada çıkıyor.** Güneş üretimi öğle saatlerinde tüketimi
değil ama *artık yükü* çökertir. Sonuç: tüketim öğlen zirvedeyken PTF çukur
yapabilir — yani fiyat profili tüketim profilini değil, **artık yük profilini**
takip eder. Yukarıdaki iki korelasyon rakamı bunun sayısal karşılığı.
("Ördek eğrisi" / duck curve dediğimiz şey tam olarak bu.)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

for i, y in enumerate(years):
    v = np.sort(D.loc[D["year"] == y, "ptf"].values)[::-1]
    axes[0].plot(np.linspace(0, 100, len(v)), v, color=CAT[i % len(CAT)], label=str(y))
finish(axes[0], "2.3 · Fiyat süre eğrisi (PDC)",
       "Yükün LDC'si düz inerken fiyatınki L şeklinde — merit order dışbükeyliği",
       "TL/MWh", "Yılın yüzdesi (%)")

tab = pd.DataFrame({
    "Ortalama":  D.groupby("year")["ptf"].mean(),
    "Medyan":    D.groupby("year")["ptf"].median(),
    "P95":       D.groupby("year")["ptf"].quantile(.95),
    "Maks":      D.groupby("year")["ptf"].max(),
    "Sıfır saat":D.assign(z=D["ptf"] <= 0.01).groupby("year")["z"].sum(),
    "Tavanda %": D.assign(t=D["ptf"] >= D.groupby("year")["ptf"].transform("max")*0.999)
                  .groupby("year")["t"].mean()*100,
}).round(1)
axes[1].axis("off")
axes[1].table(cellText=tab.values, rowLabels=tab.index.astype(str),
              colLabels=tab.columns, loc="center", cellLoc="center").scale(1, 1.5)
axes[1].set_title("2.3b · Yıllık fiyat istatistikleri", loc="left")
plt.tight_layout()
display(tab)

**PDC'nin L şekli merit order'ın imzasıdır.** Yük süre eğrisi yumuşak inerken
fiyat süre eğrisi keskin bir dirsek yapar: saatlerin çoğunda fiyat dar bir bantta
(marjinal santral hep aynı teknoloji), yılın küçük bir diliminde ise patlar
(pahalı pik üniteler devrede). Sıfır fiyat saatleri ise madalyonun diğer yüzü:
yenilenebilir bolluğu + düşük talep.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

axes[0].hist(D["ptf"], bins=80, color=CAT[0], alpha=.85)
finish(axes[0], "2.4 · PTF dağılımı", None, "Saat sayısı", "TL/MWh", legend=False)

pos = D.loc[D["ptf"] > 0, "ptf"]
axes[1].hist(np.log(pos), bins=80, color=CAT[2], alpha=.85)
finish(axes[1], "2.4b · log(PTF)", "Log dönüşüm sonrası simetriye yaklaşıyor mu?",
       "Saat sayısı", "log TL/MWh", legend=False)

vol = D.groupby(D.index.to_period("M"))["ptf"].std() / D.groupby(D.index.to_period("M"))["ptf"].mean()
axes[2].plot(vol.index.to_timestamp(), vol.values, color=CAT[3], lw=2)
finish(axes[2], "2.4c · Aylık oynaklık (CV)", "std / ortalama", "Değişim katsayısı",
       legend=False)
plt.tight_layout()
print(f"Çarpıklık: {D['ptf'].skew():.2f} · Basıklık: {D['ptf'].kurtosis():.2f} "
      f"(normal dağılımda 0 ve 0)")

Kalın sağ kuyruk = fiyat spike'ları. Bu, model seçimini doğrudan etkiler:
* Ham PTF üzerinde MSE minimize eden bir model kuyruğa aşırı ağırlık verir.
* **log dönüşüm** ya da **kuantil regresyon** (P10/P50/P90) çoğu zaman daha iyi.
* Spike saatlerini ayrı bir sınıflandırma problemi olarak ele almak da bir seçenek.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))
sp = D.groupby("hour")["spread"].agg(["mean", lambda s: s.quantile(.25), lambda s: s.quantile(.75)])
sp.columns = ["mean", "q25", "q75"]
axes[0].fill_between(sp.index, sp["q25"], sp["q75"], color=CAT[1], alpha=.20, label="Ç1–Ç3")
axes[0].plot(sp.index, sp["mean"], color=CAT[1], marker="o", ms=3, label="Ortalama")
axes[0].axhline(0, color=MUTED, lw=1, ls="--")
finish(axes[0], "2.5 · SMF − PTF farkı (saate göre)",
       "Pozitif → sistem enerji açığında (yukarı yönlü talimat pahalı)",
       "TL/MWh", "Saat")
axes[0].set_xticks(range(0, 24, 3))

ms = D.groupby(D.index.to_period("M"))["spread"].mean()
axes[1].bar(ms.index.to_timestamp(), ms.values, width=22,
            color=np.where(ms.values > 0, CAT[3], CAT[0]))
axes[1].axhline(0, color=MUTED, lw=1)
finish(axes[1], "2.5b · Aylık ortalama SMF − PTF",
       "Kırmızı: sistem enerji açığı · Mavi: fazla", "TL/MWh", legend=False)
plt.tight_layout()

**PTF vs SMF neden önemli?** PTF gün öncesinde, *beklentilerle* oluşur. SMF ise
gerçek zamanda, sistemin fiilen dengelendiği yerde. Aradaki fark sistematik
olarak pozitifse gün öncesi piyasası talebi/kıtlığı sürekli **eksik**
fiyatlıyordur. Bu fark, tahmin modelinde "PTF'nin nereye kayacağı" için sinyal
taşıyabilir ve dengesizlik maliyetini doğrudan belirler.

---
# 3. Arz-Talep Dengesi — Merit Order'ı Veriden Çıkarmak

Buraya kadar talebi ve fiyatı ayrı ayrı gördük. Şimdi ikisini bağlayan mekanizmayı
kuruyoruz.

**Teori, iki cümlede:** Gün öncesi piyasada her santral kendi kısa dönem marjinal
maliyeti (SRMC) üzerinden teklif verir. Piyasa işletmecisi teklifleri ucuzdan
pahalıya dizer (**merit order**) ve talebi karşılayana kadar sırayla kabul eder.
Talebi karşılayan **son** santralin teklifi herkesin aldığı fiyat olur — PTF.

Sonuç: **PTF ≈ marjinal santralin SRMC'si.** Dolayısıyla PTF'yi açıklamak için
iki şey lazım: (1) merit order'da ne kadar ilerlediğimiz → **artık yük**,
(2) o noktadaki santralin yakıt maliyeti.

In [ ]:
gen_cols = [c for c in ZERO_MC + FLEX if c in D.columns]
order = [c for c in ["wind","sun","river","geothermal","biomass","wasteheat","nuclear",
                     "dammedHydro","lignite","importCoal","blackCoal","asphaltiteCoal",
                     "naturalGas","fueloil","gasOil","lng","naphta"] if c in gen_cols]

# Örnek bir hafta: en yüksek yenilenebilir paylı haftayı seç (ilginç olan orada)
wk_share = D["ren_share"].resample("W").mean()
w0 = wk_share.idxmax() - pd.Timedelta(days=6)
sub = D.loc[w0:w0 + pd.Timedelta(days=7)]

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                         gridspec_kw={"height_ratios": [2.2, 1]})
axes[0].stackplot(sub.index, [sub[c].values/1000 for c in order],
                  labels=[TR_NAME.get(c, c) for c in order],
                  colors=[fcol(c) for c in order], alpha=.92,
                  edgecolor="white", linewidth=.4)
axes[0].plot(sub.index, sub["consumption"]/1000, color=INK, lw=2, label="Tüketim")
axes[0].plot(sub.index, sub["residual"]/1000, color=INK, lw=1.6, ls="--", label="Artık yük")
axes[0].legend(ncols=8, fontsize=7.5, loc="lower left", bbox_to_anchor=(0, 1.015),
               columnspacing=1.1, handlelength=1.2)
axes[0].set_title(f"3.1 · Üretim yığını — {w0:%d %b %Y} haftası (en yüksek YEK paylı hafta)",
                  loc="left", pad=52)
axes[0].set_ylabel("GW")

axes[1].plot(sub.index, sub["ptf"], color=CAT[3], lw=2)
axes[1].set_ylabel("PTF (TL/MWh)"); axes[1].set_xlabel("")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[1].set_title("Aynı haftanın PTF'si — yığın kalınlaştıkça fiyat ne yapıyor?",
                  loc="left", fontsize=10, fontweight="normal", color=MUTED)
plt.tight_layout()

Bu iki panel yan yana okunmalı: yeşil/mavi (rüzgâr-güneş) bandı kalınlaştığında
üstteki siyah kesikli çizgi (**artık yük**) çöküyor ve alttaki fiyat onunla
birlikte iniyor. Aynı saatte **tüketim** (düz siyah) değişmemiş olabilir. İşte
"fiyat arz-talep dengesini yansıtır" cümlesinin operasyonel hâli bu.

In [ ]:
mix = (D[order].resample(FREQ_Y).mean() / D[order].resample(FREQ_Y).mean().sum(axis=1).values[:, None] * 100)
fig, ax = plt.subplots(figsize=(11, 4.9))
bottom = np.zeros(len(mix))
for i, c in enumerate(order):
    ax.bar(mix.index.year.astype(str), mix[c], bottom=bottom,
           color=fcol(c), label=TR_NAME.get(c, c), width=.62,
           edgecolor="white", linewidth=1.0)
    for xi, (b, v) in enumerate(zip(bottom, mix[c].values)):
        if v > 6: ax.text(xi, b + v/2, f"{v:.0f}", ha="center", va="center",
                          fontsize=8, color="white", fontweight="bold")
    bottom += mix[c].values
ax.legend(ncols=6, fontsize=8, loc="upper center", bbox_to_anchor=(.5, -0.13),
          columnspacing=1.1, handlelength=1.2)
finish(ax, "3.2 · Yıllık üretim kompozisyonu (%)",
       "Merit order'ın yıllar içinde nasıl kaydığı — YEK payı arttıkça "
       "aynı talep daha ucuz karşılanır", "%", legend=False)

## 3.3 Asıl grafik: fiyat neye karşı çiziliyor?

Şimdi kritik karşılaştırma. Solda **PTF vs brüt tüketim**, sağda **PTF vs artık
yük**. İkisi de aynı veriden, aynı saatlerden. Tek fark x ekseninde neyi
kullandığımız.

In [ ]:
def hexscatter(ax, x, y, title, sub, xlab, bins=70):
    hb = ax.hexbin(x, y, gridsize=bins, cmap=SEQ, mincnt=1, bins="log", linewidths=0)
    # koşullu medyan + kuantil bandı
    q = pd.DataFrame({"x": x, "y": y}).dropna()
    q["b"] = pd.qcut(q["x"], 40, duplicates="drop")
    g = q.groupby("b", observed=True).agg(x=("x", "median"), m=("y", "median"),
                                          lo=("y", lambda s: s.quantile(.25)),
                                          hi=("y", lambda s: s.quantile(.75)))
    ax.fill_between(g["x"], g["lo"], g["hi"], color=CAT[3], alpha=.18, zorder=3)
    ax.plot(g["x"], g["m"], color=CAT[3], lw=2.6, zorder=4, label="Koşullu medyan")
    r = np.corrcoef(q["x"], q["y"])[0, 1]
    rs = q["x"].corr(q["y"], method="spearman")
    finish(ax, title, f"{sub}   ·   Pearson r={r:.2f} · Spearman ρ={rs:.2f}",
           "PTF (TL/MWh)", xlab)
    return hb, r, rs

yr_focus = years[-2] if len(years) > 1 else years[-1]
Dy = D[D["year"] == yr_focus].dropna(subset=["ptf", "consumption", "residual"])

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
_, r1, s1 = hexscatter(axes[0], Dy["consumption"]/1000, Dy["ptf"],
                       f"3.3 · PTF vs BRÜT TÜKETİM ({yr_focus})",
                       "Sezgisel ama eksik", "Tüketim (GW)")
_, r2, s2 = hexscatter(axes[1], Dy["residual"]/1000, Dy["ptf"],
                       f"3.3b · PTF vs ARTIK YÜK ({yr_focus})",
                       "Piyasanın gerçekten gördüğü talep", "Artık yük (GW)")
plt.tight_layout()
print(f"Korelasyon artışı: {r1:.3f} → {r2:.3f}  (Spearman {s1:.3f} → {s2:.3f})")

**Sağdaki kırmızı çizgi, piyasanın ampirik arz eğrisidir.** Merit order'ı
santral santral bilmenize gerek yok — piyasa onu her saat yeniden fiyatlıyor,
biz sadece gözlemliyoruz. Eğrinin **dışbükeyliği** (sağa doğru dikleşmesi)
merit order'ın kendisi: ucuz üniteler tükendikçe bir sonraki MWh giderek
pahalılaşıyor.

Saçılım bandının (Ç1–Ç3) genişliği ise "aynı artık yükte fiyat neden farklı
olabiliyor?" sorusunun cevabı — yakıt fiyatı, hidro rezervuar durumu, ithalat,
arıza, teklif stratejisi. Modelde açıklanacak asıl bakiye orada.

In [ ]:
# Arz eğrisini yıllara ayırıp kaymayı gör
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
for i, y in enumerate(years):
    d = D[D["year"] == y].dropna(subset=["ptf", "residual"])
    if len(d) < 500: continue
    b = pd.qcut(d["residual"], 30, duplicates="drop")
    g = d.groupby(b, observed=True).agg(x=("residual", "median"), m=("ptf", "median"))
    axes[0].plot(g["x"]/1000, g["m"], color=CAT[i % len(CAT)], marker="o", ms=3, label=str(y))
finish(axes[0], "3.4 · Ampirik arz eğrisi — yıllara göre",
       "Dikey kayma = yakıt maliyeti/kur · Şekil değişimi = kapasite karması",
       "PTF (TL/MWh)", "Artık yük (GW)")

# Yakıt maliyetinden arındırmak için her yılın medyanına normalize et
for i, y in enumerate(years):
    d = D[D["year"] == y].dropna(subset=["ptf", "residual"])
    if len(d) < 500: continue
    b = pd.qcut(d["residual"], 30, duplicates="drop")
    g = d.groupby(b, observed=True).agg(x=("residual", "median"), m=("ptf", "median"))
    axes[1].plot(g["x"]/1000, g["m"]/d["ptf"].median(), color=CAT[i % len(CAT)],
                 marker="o", ms=3, label=str(y))
axes[1].axhline(1, color=MUTED, ls="--", lw=1)
finish(axes[1], "3.4b · Aynı eğri, yıl medyanına normalize",
       "Seviye etkisi çıkınca eğrinin ŞEKLİ karşılaştırılabilir hâle gelir",
       "PTF / yıl medyanı", "Artık yük (GW)")
plt.tight_layout()

İkinci panel bir teşhis aracı: seviye (yakıt maliyeti, kur, enflasyon) etkisini
böldüğümüzde geriye **eğrinin şekli** kalıyor. Şekil yıllar içinde düzleşiyorsa
sistem daha esnek/bol kapasiteli hâle gelmiş; dikleşiyorsa kıtlığa daha
duyarlı hâle gelmiş demektir.

## 3.5 Marjinal teknoloji kim?

PTF ≈ marjinal santralin SRMC'si ise, artık yük arttığında **hangi yakıtın
üretimi artıyor** sorusunun cevabı "o an marjinal olan teknoloji"dir.
Artık yükü dilimlere bölüp her dilimde yakıt bazlı üretimin nasıl değiştiğine
bakıyoruz.

In [ ]:
bins = pd.qcut(D["residual"], 20, duplicates="drop")
gb = D.groupby(bins, observed=True)
flex_present = [c for c in FLEX if c in D.columns and D[c].mean() > 50]
tab35 = gb[flex_present].mean()
tab35.index = (gb["residual"].median()/1000).round(1).values

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))
bottom = np.zeros(len(tab35))
for i, c in enumerate(flex_present):
    axes[0].fill_between(tab35.index, bottom, bottom + tab35[c].values/1000,
                         color=fcol(c), alpha=.92, label=TR_NAME.get(c, c),
                         linewidth=.6, edgecolor="white")
    bottom += tab35[c].values/1000
axes[0].plot(tab35.index, tab35.index, color=INK, ls="--", lw=1.4, label="45° (artık yük)")
finish(axes[0], "3.5 · Yığın nasıl dolduruluyor?",
       "Her dilimde esnek üretimin ortalama kırılımı", "GW", "Artık yük (GW)")

# Marjinal katkı: d(üretim)/d(artık yük)
slope = tab35.diff().div(np.diff(tab35.index, prepend=np.nan), axis=0)
slope = slope.iloc[1:]
for i, c in enumerate(flex_present):
    axes[1].plot(slope.index, slope[c].values/1000, color=fcol(c),
                 marker="o", ms=3, label=TR_NAME.get(c, c))
axes[1].axhline(0, color=MUTED, lw=1, ls="--")
finish(axes[1], "3.5b · Marjinal katkı Δüretim / Δartık yük",
       "1'e en yakın olan teknoloji o bantta MARJİNAL santraldir", "GW / GW",
       "Artık yük (GW)")
# Lejantlar eksenin içinde veriyi kapatıyor — altına al
for a in axes:
    a.legend(ncols=3, fontsize=9, loc="upper center", bbox_to_anchor=(.5, -0.20),
             columnspacing=1.2, handlelength=1.4)
plt.tight_layout()

Sağdaki grafik doğrudan "marjinal santral kim?" sorusunu cevaplıyor. Türkiye
sisteminde tipik desen: düşük artık yükte hidro/kömür, orta bantta ithal kömür,
yüksek bantta **doğalgaz** marjinal olur. Doğalgazın marjinal olduğu bantta
PTF ≈ `gaz_fiyatı / verim + değişken O&M` yaklaşımı iyi çalışır ve
**gaz fiyatı tahmin modelinin en güçlü dışsal değişkeni** hâline gelir.

## 3.6 Merit order etkisi: YEK üretimi fiyatı ne kadar düşürüyor?

Aynı artık yük seviyesinde daha çok rüzgâr+güneş olması fiyatı düşürür mü?
Basit ama etkili bir tanımlama: artık yükü ve takvimi sabitleyip (kontrol edip)
VRE üretiminin katsayısına bakmak.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def ols_report(df, ycol, xcols, label):
    d = df[[ycol] + xcols].dropna()
    X, y = d[xcols].values, d[ycol].values
    m = LinearRegression().fit(X, y)
    r2 = r2_score(y, m.predict(X))
    return {"model": label, "R²": round(r2, 3),
            **{c: round(b, 3) for c, b in zip(xcols, m.coef_)}}, m

Dm = D.dropna(subset=["ptf", "residual", "vre"]).copy()
Dm["resid_gw"] = Dm["residual"]/1000
Dm["vre_gw"]   = Dm["vre"]/1000
Dm["resid_sq"] = Dm["resid_gw"]**2
rows = []
for y in years:
    d = Dm[Dm["year"] == y]
    if len(d) < 2000: continue
    rep, _ = ols_report(d, "ptf", ["resid_gw", "resid_sq", "vre_gw"], str(y))
    rep["Yıl ort. PTF"] = round(d["ptf"].mean(), 1)
    rep["MOE (%/GW)"] = round(100*rep["vre_gw"]/d["ptf"].mean(), 2)
    rows.append(rep)
moe = pd.DataFrame(rows).set_index("model")
display(moe)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(moe.index, moe["MOE (%/GW)"], color=CAT[2], width=.55)
for xi, v in enumerate(moe["MOE (%/GW)"]):
    ax.text(xi, v, f"{v:.2f}%", ha="center",
            va="top" if v < 0 else "bottom", fontsize=9, color=INK)
ax.axhline(0, color=MUTED, lw=1)
ax.margins(y=0.18)
finish(ax, "3.6 · Merit order etkisi",
       "Artık yük sabitken +1 GW rüzgâr/güneşin PTF'ye etkisi (yıl ortalamasının %'si)",
       "% / GW", "Yıl", legend=False)

Katsayı negatifse (beklenen) YEK üretimi merit order'ı sola kaydırıyor ve fiyatı
baskılıyor demektir. Etkinin **büyüklüğü** yıllar içinde artıyorsa sistem
YEK'e daha duyarlı hâle gelmiştir — bu, tahmin modelinde rüzgâr/güneş
tahminlerinin ağırlığının artması gerektiği anlamına gelir.

> **Dikkat:** Bu bir *korelasyon tanımlaması*, nedensellik iddiası değil. VRE
> üretimi hava ile, hava da talep ile ilişkili. Ciddi bir nedensellik analizi
> için hava değişkenlerini kontrol etmek (ya da rüzgâr tahminini araç değişken
> olarak kullanmak) gerekir.

## 3.7 Yedek marj ve fiyat spike'ları

Merit order'ın sağ ucu: emre amade kapasite (EAK) artık yüke yaklaştıkça fiyat
doğrusal değil, **patlayarak** artar.

In [ ]:
if "margin" in D.columns:
    d = D.dropna(subset=["margin", "ptf"]).copy()
    d["mbin"] = pd.qcut(d["margin"], 25, duplicates="drop")
    g = d.groupby("mbin", observed=True).agg(
        m=("margin", "median"), p50=("ptf", "median"),
        p90=("ptf", lambda s: s.quantile(.90)), p99=("ptf", lambda s: s.quantile(.99)))
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.plot(g["m"]*100, g["p50"], color=CAT[0], marker="o", ms=3, label="Medyan PTF")
    ax.plot(g["m"]*100, g["p90"], color=CAT[1], marker="s", ms=3, label="P90")
    ax.plot(g["m"]*100, g["p99"], color=CAT[3], marker="^", ms=3, label="P99")
    ax.invert_xaxis()
    finish(ax, "3.7 · Yedek marj daraldıkça fiyat dağılımı",
           "Yedek marj = (EAK − artık yük) / EAK · sağdan sola sistem sıkışıyor",
           "PTF (TL/MWh)", "Yedek marj (%)")

Medyan çizgisi yatayken P90/P99'un yukarı fırlaması tipik: **kıtlık riski
ortalamayı değil kuyruğu hareket ettirir.** Nokta tahmini (P50) yapan bir model
bu saatlerde sistematik olarak düşük tahmin eder. Çözüm: kuantil tahmini ya da
ayrı bir spike sınıflandırıcısı.

## 3.8 Gerçek arz-talep teklif eğrileri (GÖP)

Şimdiye kadar arz eğrisini *dolaylı* çıkardık. EPİAŞ ayrıca her saat için
**gerçek teklif eğrilerini** yayınlıyor (`supply-demand`). Kesişim noktası
doğrudan PTF'yi verir — arz-talep dengesinin en birebir görüntüsü.

*(Bu hücre gerçek API bağlantısı gerektirir; `SYNTHETIC=True` iken atlanır.)*

In [ ]:
if MODE == "api":
    def supply_demand_curve(ep, when):
        res = ep.call("supply-demand", date=pd.Timestamp(when))
        items = res.get("items") or []
        sd = pd.DataFrame(items)
        print("supply-demand kolonları:", list(sd.columns))
        return sd

    # Karşılaştırma için: bir yüksek fiyatlı, bir düşük fiyatlı saat seç
    hi = D["ptf"].idxmax(); lo = D.loc[D["ptf"] > 0, "ptf"].idxmin()
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
    for ax, when, lbl in [(axes[0], hi, "En pahalı saat"), (axes[1], lo, "En ucuz saat")]:
        sd = supply_demand_curve(ep, when)
        pcol = next((c for c in ("price", "fiyat") if c in sd.columns), sd.columns[0])
        scol = next((c for c in ("supply", "arz") if c in sd.columns), None)
        dcol = next((c for c in ("demand", "talep") if c in sd.columns), None)
        ax.plot(sd[scol]/1000, sd[pcol], color=CAT[0], lw=2, label="Arz")
        ax.plot(sd[dcol]/1000, sd[pcol], color=CAT[3], lw=2, label="Talep")
        ax.axhline(D.loc[when, "ptf"], color=INK, ls="--", lw=1.2,
                   label=f"PTF = {D.loc[when,'ptf']:.0f}")
        finish(ax, f"3.8 · {lbl}: {when:%d %b %Y %H:00}",
               "Kesişim = PTF · Talep eğrisinin dikliği = fiyat esnekliği",
               "Fiyat (TL/MWh)", "Miktar (GWh)")
    plt.tight_layout()
else:
    print(f'MODE="{MODE}" — gerçek teklif eğrileri yalnızca MODE="api" ile çizilir '
          '(bu seri CSV olarak dışa aktarılamıyor).')

Bu grafiği gerçek veriyle çizdiğinde bakılacaklar:
* **Talep eğrisinin dikliği** — ne kadar dikse talep o kadar inelastik; küçük
  arz kaymaları büyük fiyat sıçraması yaratır.
* **Arz eğrisindeki basamaklar** — her basamak bir teknoloji kümesi. Kesişimin
  hangi basamakta olduğu marjinal teknolojiyi doğrudan söyler.
* **Yatay platolar** — fiyattan bağımsız (must-run / YEKDEM) teklifler.

---
# 4. Tanı: Fiyat Arz-Talep Dengesini Ne Kadar Yansıtıyor?

Şimdi baştaki yorumu **sayısal olarak** test ediyoruz. Değişkenleri kademeli
ekleyip PTF'nin varyansının ne kadarını açıkladıklarına bakıyoruz. Bu tablo
aynı zamanda tahmin modelinin özellik listesinin gerekçesi olacak.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer

# Log modeli sıfır fiyatlı saatleri kaldıramaz — onları AYRI ele almak gerekir.
_n0 = int((D["ptf"] <= 0.01).sum())
print(f"Sıfır (veya ~0) fiyatlı {_n0} saat log modelinden çıkarıldı "
      f"({100*_n0/len(D):.2f}%) — bunlar ayrı bir rejim, Bölüm 5'te not düşüldü.")
Dd = D[D["ptf"] > 0.01].dropna(subset=["ptf", "consumption", "residual", "vre"]).copy()
Dd["resid_gw"]  = Dd["residual"]/1000
Dd["cons_gw"]   = Dd["consumption"]/1000
Dd["vre_gw"]    = Dd["vre"]/1000
Dd["resid_sq"]  = Dd["resid_gw"]**2
Dd["resid_cu"]  = Dd["resid_gw"]**3
Dd["log_ptf"]   = np.log1p(Dd["ptf"])

CAL = ["hour", "dow", "month", "year"]

def fit_r2(num_cols, use_cal=True, target="log_ptf", label=""):
    cols = num_cols + (CAL if use_cal else [])
    d = Dd[cols + [target]].dropna()
    pre = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
          [c for c in CAL if c in cols])], remainder="passthrough")
    mdl = make_pipeline(pre, LinearRegression())
    mdl.fit(d[cols], d[target])
    return {"Model": label, "Değişkenler": ", ".join(num_cols) or "—",
            "R²": round(r2_score(d[target], mdl.predict(d[cols])), 3)}

steps = [
    fit_r2([], True,                                             label="M0 · sadece takvim"),
    fit_r2(["cons_gw"], True,                                    label="M1 · + brüt tüketim"),
    fit_r2(["resid_gw"], True,                                   label="M2 · + artık yük"),
    fit_r2(["resid_gw", "resid_sq", "resid_cu"], True,           label="M3 · + artık yük (kübik)"),
    fit_r2(["resid_gw", "resid_sq", "resid_cu", "vre_gw"], True, label="M4 · + VRE (merit order etkisi)"),
]
if "margin" in Dd.columns:
    Dd["margin_inv"] = 1/np.clip(Dd["margin"], 0.02, None)
    steps.append(fit_r2(["resid_gw","resid_sq","resid_cu","vre_gw","margin_inv"], True,
                        label="M5 · + yedek marj (kıtlık)"))
diag = pd.DataFrame(steps).set_index("Model")
diag["ΔR²"] = diag["R²"].diff().round(3)
display(diag)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.barh(diag.index[::-1], diag["R²"][::-1], color=CAT[0], height=.6)
for i, v in enumerate(diag["R²"][::-1]):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=9, color=INK)
ax.set_xlim(0, 1)
finish(ax, "4.1 · log(PTF) varyansının ne kadarı açıklanıyor?",
       "Kademeli model karşılaştırması (örneklem içi)", None, "R²", legend=False)

**Bu tablonun okunuşu:**
* `M1 → M2` sıçraması, "brüt tüketim yerine artık yük" tercihinin karşılığı.
  Sıçrama büyükse merit order hikâyesi veride gerçekten var demektir.
* `M2 → M3`, doğrusal olmayan (dışbükey) arz eğrisinin katkısı.
* `M4`, VRE'nin artık yük dışındaki ek etkisi (teklif davranışı, YEKDEM).
* `M5`, kıtlık priminin katkısı.
* Geriye kalan `1 − R²`: yakıt fiyatı, hidro rezervuar, ithalat/ihracat, arıza,
  stratejik teklif — yani **bir sonraki adımda toplanacak veriler.**

> Örneklem içi R² iyimserdir; Bölüm 5'te zaman-serisi CV ile dürüst ölçüm yapıyoruz.

In [ ]:
# Model hatası nerede yoğunlaşıyor? — artan analizi
cols = ["resid_gw","resid_sq","resid_cu","vre_gw"] + (["margin_inv"] if "margin_inv" in Dd else []) + CAL
d = Dd[cols + ["log_ptf"]].dropna()
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAL)],
                        remainder="passthrough")
best = make_pipeline(pre, LinearRegression()).fit(d[cols], d["log_ptf"])
d = d.assign(resid_err=d["log_ptf"] - best.predict(d[cols]))

fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))
axes[0].plot(d.index, d["resid_err"].rolling(24*30, center=True).mean(), color=CAT[3], lw=2)
axes[0].axhline(0, color=MUTED, ls="--", lw=1)
finish(axes[0], "4.2 · Artıkların zaman içindeki seyri",
       "Sürüklenme varsa açıklanmayan bir rejim var", "log-hata", legend=False)

# Ortalama hata, saat kuklaları yüzünden tanım gereği ~0 olur; asıl bilgi YAYILIMDA.
hh = d.groupby("hour")["resid_err"].agg(
    p10=lambda s: s.quantile(.10), p50="median", p90=lambda s: s.quantile(.90), sd="std")
axes[1].fill_between(hh.index, hh["p10"], hh["p90"], color=CAT[0], alpha=.20, label="P10–P90")
axes[1].plot(hh.index, hh["p50"], color=CAT[0], marker="o", ms=3, label="Medyan")
axes[1].plot(hh.index, hh["sd"], color=CAT[3], lw=2, ls="--", label="Std. sapma")
axes[1].axhline(0, color=MUTED, lw=1)
finish(axes[1], "4.2b · Saate göre hata YAYILIMI",
       "Ortalama, saat kuklaları yüzünden tanım gereği ~0 — bakılacak olan genişlik",
       "log-hata", "Saat")
axes[1].set_xticks(range(0, 24, 3))

axes[2].scatter(d["resid_gw"], d["resid_err"], s=1.5, alpha=.06, color=CAT[0])
axes[2].axhline(0, color=MUTED, ls="--", lw=1)
finish(axes[2], "4.2c · Artık yüke göre hata",
       "Sistematik eğrilik kalmışsa fonksiyon formu yetersiz", "log-hata",
       "Artık yük (GW)", legend=False)
plt.tight_layout()

---
# 5. Tahminlemeye Köprü

Analizden modele geçerken tek bir kural her şeyden önemli:

> ## Sızıntı yok.
> Gün öncesi piyasa **saat 12:00'de** kapanır ve ertesi günün 24 saati birden
> fiyatlanır. Yani `2026-03-10` gününün PTF'sini tahmin ederken elinde ancak
> `2026-03-09 12:00`'a kadar olan bilgi vardır.

Bu, yukarıdaki analizde kullandığımız değişkenlerin çoğunun **doğrudan
kullanılamayacağı** anlamına gelir: gerçek zamanlı tüketim, gerçek zamanlı
üretim, gerçekleşen artık yük — hepsi *sonradan* bilinir. Onların yerine
**ex-ante ikizlerini** koyuyoruz:

| Analizde kullandık | Tahminde yerine geçen | Yayın zamanı |
|---|---|---|
| Gerçek zamanlı tüketim | **YTP** (yük tahmin planı) | Gün öncesi |
| Gerçek zamanlı üretim (yakıt bazlı) | **KGÜP** (kesinleşmiş günlük üretim programı) | Gün öncesi 13:00 |
| Gerçekleşen rüzgâr | **Rüzgâr üretim tahmini** (`wind-forecast`) | Gün öncesi |
| EAK / yedek marj | **EAK** (emre amade kapasite) | Gün öncesi |
| Gerçekleşen artık yük | **Planlanan artık yük** = YTP − (KGÜP YEK + rüzgâr tahmini) | Gün öncesi |
| — | PTF gecikmeleri (D-1, D-2, D-7 aynı saat) | Zaten geçmiş |

In [ ]:
def build_features(D, horizon_lag_hours=36):
    """
    Sızıntısız özellik seti. Gerçek projede analiz kolonlarını KGÜP/YTP/EAK ile
    değiştir; burada iskeleti kuruyoruz. horizon_lag_hours: gün öncesi kapanışına
    göre en yakın kullanılabilir geçmiş (güvenli taraf: 36 saat).
    """
    F = pd.DataFrame(index=D.index)

    # --- Ex-ante talep ve arz göstergeleri (gün öncesinde bilinir)
    F["lep"]        = D["lep"]                      # YTP
    F["lep_gw"]     = D["lep"]/1000
    # Gerçek projede: KGÜP'ten gelen rüzgâr+güneş+akarsu programı
    F["vre_fc"]     = D["vre"]                      # >>> rüzgâr/güneş TAHMİNİ ile değiştir
    F["resid_plan"] = (D["lep"] - D["renew"])/1000  # >>> YTP − KGÜP-YEK ile değiştir
    F["resid_sq"]   = F["resid_plan"]**2
    F["resid_cu"]   = F["resid_plan"]**3
    if "eak_total" in D: F["margin_plan"] = (D["eak_total"] - (D["lep"] - D["renew"]))/D["eak_total"]

    # --- Fiyat gecikmeleri (gerçekten geçmişte olanlar)
    for lag in (24, 48, 72, 168, 336):
        F[f"ptf_lag{lag}"] = D["ptf"].shift(lag)
    F["ptf_ma168"] = D["ptf"].shift(horizon_lag_hours).rolling(168).mean()
    F["ptf_ma720"] = D["ptf"].shift(horizon_lag_hours).rolling(720).mean()
    F["ptf_same_hour_7d"] = D["ptf"].shift(168)
    # Aynı saatin son 7 günlük medyanı
    F["ptf_hour_med7"] = (D["ptf"].shift(24).groupby(D.index.hour)
                          .transform(lambda s: s.rolling(7, min_periods=3).median()))

    # --- Takvim (döngüsel kodlama, ağaç modelleri için ham hâli de faydalı)
    h, dow, doy = D.index.hour, D.index.dayofweek, D.index.dayofyear
    F["hour"], F["dow"], F["month"] = h, dow, D.index.month
    F["sin_h"], F["cos_h"] = np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)
    F["sin_d"], F["cos_d"] = np.sin(2*np.pi*doy/365), np.cos(2*np.pi*doy/365)
    F["is_wknd"] = (dow >= 5).astype(int)
    # TODO: resmî tatil + dinî bayram takvimi (Türkiye'de yük üzerinde çok güçlü etki)

    y = D["ptf"]
    keep = F.dropna().index.intersection(y.dropna().index)
    return F.loc[keep], y.loc[keep]

X, y = build_features(D)
print(f"Özellik matrisi: {X.shape}  ·  Hedef: {y.shape}")
print("Özellikler:", list(X.columns))

## 5.1 Dürüst değerlendirme: genişleyen pencere + boşluk

Zaman serisinde rastgele k-fold **yasak** — geleceği görüp geçmişi tahmin
etmiş olursun. Doğru kurgu: geçmişle eğit, ileri bir bloğu tahmin et, pencereyi
kaydır. Ayrıca eğitim ile test arasına bir **boşluk** (gap) koyuyoruz ki
gecikmeli özellikler test dönemine sızmasın.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

def smape(a, f):
    d = (np.abs(a) + np.abs(f)) / 2
    return np.mean(np.abs(a - f) / np.where(d == 0, np.nan, d)) * 100

def pinball(a, f, q):
    e = a - f
    return np.mean(np.maximum(q*e, (q-1)*e))

def walk_forward(X, y, n_folds=6, test_days=30, gap_hours=48):
    """Genişleyen pencere; her fold son test_days günü tahmin eder."""
    idx = X.index
    end = idx.max()
    folds = []
    for k in range(n_folds, 0, -1):
        t1 = end - pd.Timedelta(days=test_days*(k-1))
        t0 = t1 - pd.Timedelta(days=test_days)
        tr = idx[idx < t0 - pd.Timedelta(hours=gap_hours)]
        te = idx[(idx >= t0) & (idx < t1)]
        if len(tr) > 24*180 and len(te) > 24*7:
            folds.append((tr, te))
    return folds

MODELS = {
    "B1 · Naive D-7 (aynı saat)": None,          # özel işlem
    "B2 · Saat×gün ortalaması":   None,
    "M1 · Ridge":                 Ridge(alpha=5.0),
    "M2 · Gradient Boosting":     HistGradientBoostingRegressor(
                                     max_iter=400, learning_rate=0.06,
                                     max_depth=None, min_samples_leaf=40,
                                     l2_regularization=1.0, random_state=0),
}

folds = walk_forward(X, y)
if not folds:
    raise SystemExit(
        "Bu bölüm için veri yetersiz: genişleyen pencere CV en az ~7 aylık eğitim + "
        "1 aylık test istiyor. START_DATE'i geriye çekip tekrar dene.")
print(f"{len(folds)} fold · test uzunluğu {len(folds[0][1])} saat\n")

results = []
preds_store = {}
for name, mdl in MODELS.items():
    maes, smapes, allp, allt = [], [], [], []
    for tr, te in folds:
        if name.startswith("B1"):
            p = y.reindex(te - pd.Timedelta(days=7)).values
            p = pd.Series(p, index=te).ffill().bfill().values
        elif name.startswith("B2"):
            base = y.loc[tr].groupby([tr.hour, tr.dayofweek]).mean()
            p = np.array([base.get((t.hour, t.dayofweek), y.loc[tr].mean()) for t in te])
        else:
            m = mdl.fit(X.loc[tr], np.log1p(y.loc[tr]))
            p = np.expm1(m.predict(X.loc[te]))
        a = y.loc[te].values
        maes.append(mean_absolute_error(a, p)); smapes.append(smape(a, p))
        allp.append(pd.Series(p, index=te)); allt.append(y.loc[te])
    preds_store[name] = (pd.concat(allp), pd.concat(allt))
    results.append({"Model": name, "MAE (TL/MWh)": np.mean(maes),
                    "sMAPE (%)": np.mean(smapes), "MAE std": np.std(maes)})

res = pd.DataFrame(results).set_index("Model").round(2)
res["MAE iyileşme %"] = (100*(1 - res["MAE (TL/MWh)"]/res["MAE (TL/MWh)"].iloc[0])).round(1)
display(res)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2))
axes[0].barh(res.index[::-1], res["MAE (TL/MWh)"][::-1], color=CAT[0], height=.55,
             xerr=res["MAE std"][::-1], error_kw=dict(ecolor=MUTED, lw=1, capsize=3))
for i, v in enumerate(res["MAE (TL/MWh)"][::-1]):
    axes[0].text(v, i, f" {v:.0f}", va="center", fontsize=9, color=INK)
finish(axes[0], "5.1 · Fold'lar arası ortalama MAE",
       "Hata çubuğu = fold'lar arası std (istikrar göstergesi)", None,
       "TL/MWh", legend=False)

bm = list(preds_store)[-1]
p, a = preds_store[bm]
last = a.index >= a.index.max() - pd.Timedelta(days=21)
axes[1].plot(a.index[last], a[last].values, color=INK, lw=1.6, label="Gerçekleşen")
axes[1].plot(p.index[last], p[last].values, color=CAT[3], lw=1.6, label="Tahmin")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
finish(axes[1], f"5.1b · {bm} — son 3 hafta", None, "TL/MWh")
plt.tight_layout()

In [ ]:
# Hata nerede yoğunlaşıyor? — saat ve fiyat seviyesine göre
p, a = preds_store[bm]
err = pd.DataFrame({"a": a, "p": p}).assign(ae=lambda d: (d.a - d.p).abs())
fig, axes = plt.subplots(1, 2, figsize=(13.5, 3.8))
hh = err.groupby(err.index.hour)["ae"].mean()
axes[0].bar(hh.index, hh.values, color=CAT[0])
finish(axes[0], "5.2 · Saate göre MAE", None, "TL/MWh", "Saat", legend=False)

err["dec"] = pd.qcut(err["a"], 10, labels=[f"D{i+1}" for i in range(10)])
dd = err.groupby("dec", observed=True)["ae"].mean()
axes[1].bar(dd.index.astype(str), dd.values, color=CAT[3])
finish(axes[1], "5.2b · Gerçekleşen fiyat desiline göre MAE",
       "En pahalı desil (D10) hatanın çoğunu taşıyorsa spike problemi vardır",
       "TL/MWh", "Fiyat desili", legend=False)
plt.tight_layout()

## 5.3 Kuantil tahmini — nokta tahmini yetmez

Bölüm 2.4'te gördüğümüz kalın kuyruk yüzünden tek bir sayı vermek yanıltıcı.
Ticari kararlarda (teklif stratejisi, depolama çevrimi, dengesizlik riski)
aslında **dağılım** lazım. `HistGradientBoostingRegressor(loss="quantile")`
ile P10/P50/P90 üretiyoruz.

In [ ]:
tr, te = folds[-1]
qmodels, qpred = {}, {}
for q in (0.1, 0.5, 0.9):
    m = HistGradientBoostingRegressor(loss="quantile", quantile=q, max_iter=350,
                                      learning_rate=0.06, min_samples_leaf=40,
                                      random_state=0).fit(X.loc[tr], y.loc[tr])
    qmodels[q] = m; qpred[q] = pd.Series(m.predict(X.loc[te]), index=te)

act = y.loc[te]
cov = ((act >= qpred[0.1]) & (act <= qpred[0.9])).mean()*100
pb  = np.mean([pinball(act.values, qpred[q].values, q) for q in (0.1, 0.5, 0.9)])

fig, ax = plt.subplots(figsize=(13, 4.2))
w = te[te >= te.max() - pd.Timedelta(days=12)]
ax.fill_between(w, qpred[0.1].loc[w], qpred[0.9].loc[w], color=CAT[0], alpha=.20,
                label="P10–P90 aralığı")
ax.plot(w, qpred[0.5].loc[w], color=CAT[0], lw=1.8, label="P50 tahmin")
ax.plot(w, act.loc[w], color=INK, lw=1.6, label="Gerçekleşen")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
finish(ax, "5.3 · Kuantil tahmini (son fold)",
       f"P10–P90 kapsama: {cov:.1f}% (hedef ~80%) · Ortalama pinball: {pb:.1f}",
       "TL/MWh")
print(f"Kapsama {cov:.1f}% — hedefin ALTINDAysa model aşırı özgüvenli: bantlar dar, "
      f"spike saatlerinde gerçekleşen bandın dışına çıkıyor.\n"
      f"Düzeltme yolu: konformal kalibrasyon (bir ayırma kümesinde bantları "
      f"nominal kapsamayı tutturacak kadar genişlet).")

In [ ]:
# Hangi değişkenler taşıyor? — permütasyon önemi (gerçek nedensellik değil, sinyal göstergesi)
from sklearn.inspection import permutation_importance
m = MODELS["M2 · Gradient Boosting"].fit(X.loc[tr], np.log1p(y.loc[tr]))
pi = permutation_importance(m, X.loc[te], np.log1p(y.loc[te]), n_repeats=5,
                            random_state=0, scoring="neg_mean_absolute_error")
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.barh(imp.index[-14:], imp.values[-14:], color=CAT[2], height=.65)
finish(ax, "5.4 · Permütasyon önemi (son fold)",
       "Özelliği karıştırınca MAE ne kadar kötüleşiyor?", None, "Δ MAE (log ölçek)",
       legend=False)

---
# 6. Buradan Nereye? — Çalışmayı Derinleştirme Sırası

### Aşama A — Veri tabanını genişlet (en yüksek getiri burada)
Şu an `1 − R²`'nin içinde saklı olan şeyler:

1. **Doğalgaz referans fiyatı** — gaz marjinal santral olduğu saatlerde PTF'yi
   neredeyse bire bir belirler. BOTAŞ tarifesi, TTF/Henry Hub, EPİAŞ doğalgaz
   piyasası (`natural-gas-service/v1/...`) alternatifleri.
2. **Hava durumu** — sıcaklık (HDD/CDD dönüşümüyle), rüzgâr hızı, ışınım.
   Talebin ve VRE'nin ortak sürücüsü. Open-Meteo ücretsiz ve geriye dönük.
3. **Hidro rezervuar doluluk** — barajlı hidronun fırsat maliyetini belirler;
   kurak yıllarda merit order'ı yukarı iter.
4. **Sınır ötesi ticaret** — ithalat/ihracat kapasitesi ve akışı.
5. **Arıza/bakım bildirimleri (UEVM, KGÜP revizyonları)** — arz şoklarının kaynağı.
6. **Tatil takvimi** — Türkiye'de resmî + dinî bayramlar yükte %10-15 kırılma yapar.

### Aşama B — Modelleme derinliği
* **Hiyerarşik / iki aşamalı kurgu:** önce artık yükü tahmin et, sonra ampirik
  arz eğrisinden (Bölüm 3.4) fiyata çevir. Fiziksel yapıyı modele gömer,
  az veriyle daha iyi genelleşir.
* **24 saatin birlikte tahmini:** GÖP tek seferde 24 saati fiyatlıyor; saatler
  bağımsız değil. Çok çıktılı model ya da saat-bazlı ayrı modeller + ortak
  özellikler.
* **Rejim farkındalığı:** tavan fiyat dönemleri, YEKDEM değişiklikleri,
  piyasa kuralı güncellemeleri için kukla/rejim değişkenleri.
* **Spike modeli:** ayrı bir sınıflandırıcı (P95 üstü mü?) + koşullu büyüklük.
* **Konformal tahmin:** kuantil bantlarının kapsamasını garantiye almak için.

### Aşama C — Değerlendirmeyi ticarileştir
İstatistiksel metrik (MAE) ile karar metriği farklı şeyler:
* **Dengesizlik maliyeti:** tahmin hatasının SMF-PTF farkıyla çarpımı — bir
  üretici/tedarikçi için gerçek maliyet.
* **Depolama arbitraj getirisi:** tahminle çalıştırılan bir batarya, mükemmel
  öngörüye kıyasla getirinin yüzde kaçını yakalıyor? (Depolama tarafında
  çalışıyorsan asıl metrik bu.)
* **Yön isabeti:** saatler arası fiyat farkının işaretini doğru bilme oranı.

### Aşama D — Bunu bir işe dönüştür
* Günlük otomatik veri çekme + model yeniden eğitme (cron/Colab scheduled).
* Tahmin vs gerçekleşen izleme paneli (drift takibi).
* Bir bildiri/rapor çıkışı: "Türkiye elektrik piyasasında merit order etkisinin
  yıllar içindeki evrimi" Bölüm 3.6'daki tablo tek başına bir bulgu.

---
### Hızlı kontrol listesi

- [ ] `SYNTHETIC = False` yap, EPİAŞ kimlik bilgilerini gir, veriyi çek
- [ ] `normalize()` çıktısındaki **ham kolon adlarını** oku — `build_master` içindeki
      `pick()` adaylarını gerçek adlarla güncelle
- [ ] Bölüm 3.3'teki iki korelasyonu karşılaştır → merit order hikâyesi veride var mı?
- [ ] Bölüm 3.8'i gerçek teklif eğrileriyle çalıştır
- [ ] Bölüm 4'teki `ΔR²` sütununa bak → hangi değişken gerçekten kazandırıyor?
- [ ] Bölüm 5'te `vre_fc` ve `resid_plan`'ı **gerçek tahmin/KGÜP verisiyle** değiştir
- [ ] Hava durumu ve gaz fiyatı verisini ekle, Bölüm 4'ü tekrar çalıştır